# Results Analysis — Multi-Tool Agent Harness for RAN Energy Optimization

Compares four experimental conditions:
- **Baseline A** — Rule-based (no LLM)
- **Baseline B** — Open-loop LLM (no tools, no validation)
- **Baseline C** — Digital twin only (Planner + Validator + VIAVI, no additional tools)
- **Full harness** — Baseline C + all 5 tools

Metrics: energy savings, QoS violation rate, decision quality, per-tool ablation.

**Demo mode**: when no `output/` results exist, synthetic plausible data is generated so all cells run end-to-end.

In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore')

# ── paths ─────────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path('..').resolve()
OUTPUT_DIR   = PROJECT_ROOT / 'output'
print(f'OUTPUT_DIR → {OUTPUT_DIR}')

# ── plot style ────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi':       150,
    'font.family':      'sans-serif',
    'font.size':        11,
    'axes.spines.top':  False,
    'axes.spines.right':False,
})

CONDITION_ORDER  = [
    'baseline_rules',
    'baseline_openloop_vtzcycle',
    'baseline_digital_twin_vtzcycle',
    'full_harness_vtzcycle__nemotron-3-super-120b-a12b',
]
CONDITION_LABELS = {
    'baseline_rules':                                    'A: Rule-based',
    'baseline_openloop_vtzcycle':                        'B: Open-loop LLM',
    'baseline_digital_twin_vtzcycle':                    'C: Digital twin',
    'full_harness_vtzcycle__nemotron-3-super-120b-a12b': 'Proposed: Full Harness',   # models added from the run in section 1
}
PALETTE = {
    'baseline_rules':                                    '#9e9e9e',
    'baseline_openloop_vtzcycle':                        '#ef9a9a',
    'baseline_digital_twin_vtzcycle':                    '#90caf9',
    'full_harness_vtzcycle__nemotron-3-super-120b-a12b': '#1565c0',
}
# Conditions with a Validator (approval rate is meaningful for these)
def has_validator(cond: str) -> bool:
    return cond.startswith(('baseline_digital_twin', 'full_harness'))

TOOL_NAMES = ['historical_kpi', 'traffic_forecast', 'alarm_fault', 'interference', 'energy_pricing']

print('✓ Imports OK')

## 1. Load results

In [ ]:
import re

def load_jsonl(path: Path) -> list[dict]:
    rows = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


MAIN_INTENT_MBPS = 5.0   # sections 2–8 compare the conditions at this intent


def _intent_mbps(cfg: dict) -> float:
    """Intent of a run from its run_config.json; runs without one ran at 5 Mbps."""
    m = re.search(r"[\d.]+", str(cfg.get('intent', '')))
    return float(m.group()) if m else MAIN_INTENT_MBPS


def load_all_results() -> pd.DataFrame:
    """Load iterations.jsonl files from output/.

    For each base condition and intent, keeps only the most recent run directory.
    Injects planner_model / validator_model from run_config.json when the
    JSONL rows are missing those fields (runs predating the provenance fix).
    """
    # Collect all run dirs with a JSONL
    run_dirs = sorted(
        (d for d in OUTPUT_DIR.glob('run_*') if (d / 'iterations.jsonl').exists()),
        key=lambda d: d.name,
    )

    def _config(d: Path) -> dict:
        p = d / 'run_config.json'
        return json.loads(p.read_text()) if p.exists() else {}

    # Keep latest run per base condition (everything after the timestamp) and
    # intent: the run name does not say the intent, run_config.json does
    latest: dict[tuple[str, float], Path] = {}
    for d in run_dirs:
        parts = d.name.split('_', 3)          # ['run', YYYYMMDD, HHMMSS, rest]
        base  = parts[3] if len(parts) > 3 else d.name
        latest[(base, _intent_mbps(_config(d)))] = d   # later sort order wins

    records = []
    for (_, intent), run_dir in latest.items():
        jsonl = run_dir / 'iterations.jsonl'
        rows  = load_jsonl(jsonl)

        # run_config.json for provenance (planner/validator model)
        cfg = _config(run_dir)

        for r in rows:
            r['run_dir'] = run_dir.name
            r['intent_mbps'] = intent
            # Backfill model fields from run_config when missing in the row
            if not r.get('planner_model') and cfg.get('planner_model'):
                r['planner_model']   = cfg['planner_model']
                r['validator_model'] = cfg.get('validator_model', '')
            records.append(r)
    return pd.DataFrame(records)


df_all_raw = load_all_results()   # every intent; section 9 uses it
df_raw = (df_all_raw[df_all_raw['intent_mbps'] == MAIN_INTENT_MBPS].copy()
          if not df_all_raw.empty else df_all_raw)
if not df_all_raw.empty:
    print('Runs per intent (Mbps):', df_all_raw.groupby('intent_mbps')['run_dir'].nunique().to_dict(),
          f'— sections 2–8 use {MAIN_INTENT_MBPS:g} Mbps')

# Name the full harness's models in its label, from the run actually loaded
_FH = 'full_harness_vtzcycle__nemotron-3-super-120b-a12b'
_fh = df_raw[df_raw['condition'] == _FH] if not df_raw.empty else df_raw
if len(_fh):
    _short = lambda m: m.split('/')[-1] if isinstance(m, str) and m else None
    _pl = _short(_fh['planner_model'].iloc[0]) if 'planner_model' in _fh else None
    _vl = _short(_fh['validator_model'].iloc[0]) if 'validator_model' in _fh else None
    _models = ' + '.join(dict.fromkeys(m for m in (_pl, _vl) if m))   # one name when both are the same
    if _models:
        CONDITION_LABELS[_FH] = f'Proposed: Full Harness ({_models})'

# The full harness with the tuned policy, at the main intent: compared in
# sections 2–8 alongside the other conditions when a run of it exists
_FH_TUNED = 'full_harness_vtzcycle_tuned_calibrated__nemotron-3-super-120b-a12b'
if not df_raw.empty and (df_raw['condition'] == _FH_TUNED).any() and _FH_TUNED not in CONDITION_ORDER:
    CONDITION_ORDER.append(_FH_TUNED)
    CONDITION_LABELS[_FH_TUNED] = 'Proposed: Full Harness, tuned policy'
    PALETTE[_FH_TUNED] = '#0d3c78'

if df_raw.empty:
    print('⚠  No results found in output/ — switching to DEMO MODE with synthetic data.')
    DEMO_MODE = True
else:
    DEMO_MODE = False
    print(f'✓ Loaded {len(df_raw):,} iteration records from {df_raw["run_dir"].nunique()} runs')
    print(df_raw['condition'].value_counts().to_string())

In [ ]:
# ── Demo mode: generate plausible synthetic results ───────────────────────────
if DEMO_MODE:
    rng = np.random.default_rng(42)
    N   = 30  # iterations per condition

    def _make_condition(condition, n_cells=6, sleep_mean=0.25, sleep_std=0.10,
                        qos_viol_prob=0.15, approval_rate=1.0, proposed_mean=2.5):
        rows = []
        for i in range(N):
            n_proposed = max(0, int(rng.normal(proposed_mean, 0.8)))
            n_approved = int(n_proposed * approval_rate)
            n_rejected = n_proposed - n_approved
            sleep_frac = np.clip(rng.normal(sleep_mean, sleep_std), 0, 1)
            rows.append({
                'iteration':          i + 1,
                'condition':          condition,
                'n_proposed':         n_proposed,
                'n_approved':         n_approved,
                'n_rejected':         n_rejected,
                'sleep_fraction':     round(sleep_frac, 3),
                'qos_violated':       int(rng.random() < qos_viol_prob),
                'mean_throughput_mbps': round(rng.normal(7.2 - sleep_frac * 1.5, 0.4), 2),
                'planner_elapsed_s':  round(rng.uniform(3, 8), 2),
                'validator_elapsed_s':round(rng.uniform(4, 10), 2) if approval_rate < 1 else 0.0,
                'total_elapsed_s':    round(rng.uniform(8, 20), 2),
                'tools_used':         [],
            })
        return rows

    synthetic = (
        _make_condition('baseline_rules',        sleep_mean=0.22, qos_viol_prob=0.20, approval_rate=1.00, proposed_mean=1.8) +
        _make_condition('baseline_openloop',     sleep_mean=0.35, qos_viol_prob=0.30, approval_rate=1.00, proposed_mean=3.2) +
        _make_condition('baseline_digital_twin', sleep_mean=0.30, qos_viol_prob=0.10, approval_rate=0.75, proposed_mean=3.0) +
        _make_condition('full_harness',          sleep_mean=0.42, qos_viol_prob=0.03, approval_rate=0.88, proposed_mean=3.5)
    )

    # Ablation runs — each tool added one at a time on top of baseline C
    ablation_params = {
        'ablation_historical_kpi':   dict(sleep_mean=0.33, qos_viol_prob=0.09),
        'ablation_traffic_forecast': dict(sleep_mean=0.34, qos_viol_prob=0.08),
        'ablation_alarm_fault':      dict(sleep_mean=0.31, qos_viol_prob=0.06),
        'ablation_interference':     dict(sleep_mean=0.32, qos_viol_prob=0.07),
        'ablation_energy_pricing':   dict(sleep_mean=0.35, qos_viol_prob=0.08),
    }
    for label, params in ablation_params.items():
        synthetic += _make_condition(label, approval_rate=0.80, proposed_mean=3.1, **params)

    df_raw = pd.DataFrame(synthetic)
    print(f'✓ Demo data: {len(df_raw)} rows across {df_raw["condition"].nunique()} conditions')

In [ ]:
# ── Normalise columns that may be absent in real results ──────────────────────

QOS_VIOLATION_THRESHOLD_MBPS = 5.0  # operator intent


def extract_sleep_fraction(row):
    # Prefer post_kpis (reflects state after this iteration's actions).
    # Fall back to pre_kpis (state before actions = same as previous post_kpis)
    # for no-action iterations where post_kpis is empty.
    for kpi_key in ('post_kpis', 'pre_kpis'):
        kpi = row.get(kpi_key, {})
        if isinstance(kpi, dict) and kpi:
            if 'sleeping_cells' in kpi and 'total_cells' in kpi:
                tc = kpi['total_cells']
                return kpi['sleeping_cells'] / tc if tc > 0 else 0.0
    return np.nan


def extract_avg_throughput(row):
    if 'mean_throughput_mbps' in row and pd.notna(row['mean_throughput_mbps']):
        return row['mean_throughput_mbps']
    # post_kpis is the ground truth when actions were applied this iteration.
    # pre_kpis covers every iteration (new runs); it is sim_test_fn([]) at the
    # start of the window and equals the previous iteration's post-action state.
    # Prefer post_kpis, then pre_kpis, then per_site fallback.
    for kpi_key in ('post_kpis', 'pre_kpis'):
        kpi = row.get(kpi_key, {})
        if not isinstance(kpi, dict):
            continue
        if 'avg_throughput_mbps' in kpi:
            return kpi['avg_throughput_mbps']
    post = row.get('post_kpis', {}) or {}
    per_site = post.get('per_site', {})
    if per_site:
        awake_qos = [
            v.get('avg_qos', np.nan)
            for v in per_site.values()
            if isinstance(v, dict) and not v.get('n1_sleeping', False)
        ]
        vals = [x for x in awake_qos if not np.isnan(x) and x > 0]
        return sum(vals) / len(vals) if vals else np.nan
    return np.nan


def extract_qos_violated(row):
    if 'qos_violated' in row and pd.notna(row['qos_violated']):
        return int(row['qos_violated'])
    tp = extract_avg_throughput(row)
    if np.isnan(tp):
        return 0
    return int(tp < QOS_VIOLATION_THRESHOLD_MBPS)


df = df_raw.copy()
df['sleep_fraction']      = df.apply(extract_sleep_fraction, axis=1)
df['avg_throughput_mbps'] = df.apply(extract_avg_throughput, axis=1)
df['qos_violated']        = df.apply(extract_qos_violated, axis=1)
df['approval_rate']       = np.where(
    df['n_proposed'] > 0,
    df['n_approved'] / df['n_proposed'],
    np.nan
)

# Forward-fill sleep_fraction and avg_throughput within each condition:
# when no actions are taken the harness skips sim_apply so post_kpis is empty —
# the network state is unchanged, so the previous iteration's values carry forward.
df = df.sort_values(['condition', 'iteration'])
for col in ['sleep_fraction', 'avg_throughput_mbps']:
    df[col] = df.groupby('condition')[col].transform(
        lambda s: s.ffill().fillna(0.0)
    )

df['is_main_condition'] = df['condition'].isin(CONDITION_ORDER)
df_main = df[df['is_main_condition']].copy()
print('Conditions in dataset:', df['condition'].unique().tolist())
print(df_main.groupby('condition')[['sleep_fraction','avg_throughput_mbps','qos_violated']].agg(
    {'sleep_fraction': 'mean', 'avg_throughput_mbps': 'mean', 'qos_violated': 'sum'}
).round(3))

## 2. Energy savings

In [ ]:
# ── 2a. Mean sleeping fraction per condition ──────────────────────────────────
energy_stats = (
    df_main.groupby('condition')['sleep_fraction']
    .agg(['mean', 'std', 'count'])
    .reindex(CONDITION_ORDER)
    .reset_index()
)
energy_stats['se'] = energy_stats['std'] / np.sqrt(energy_stats['count'])

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(
    [CONDITION_LABELS[c] for c in energy_stats['condition']],
    energy_stats['mean'] * 100,
    yerr=energy_stats['se'] * 100,
    color=[PALETTE[c] for c in energy_stats['condition']],
    capsize=4, width=0.55, error_kw={'linewidth': 1.2},
)
ax.set_ylabel('Mean cells sleeping (%)')
ax.set_title('Energy Savings by Condition')
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'analysis' / 'fig_energy_savings.pdf', bbox_inches='tight')
plt.show()
print(energy_stats[['condition','mean','se']].to_string(index=False))

In [ ]:
# ── 2b. Sleep fraction over iterations (line chart) ──────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
for cond in CONDITION_ORDER:
    sub = df_main[df_main['condition'] == cond].sort_values('iteration')
    ax.plot(sub['iteration'], sub['sleep_fraction'] * 100,
            label=CONDITION_LABELS[cond], color=PALETTE[cond],
            linewidth=1.8, alpha=0.85)

ax.set_xlabel('Iteration')
ax.set_ylabel('Cells sleeping (%)')
ax.set_title('Sleep Fraction Over Iterations')
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.legend(fontsize=9, frameon=False)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'analysis' / 'fig_sleep_over_time.pdf', bbox_inches='tight')
plt.show()

## 3. QoS preservation

In [ ]:
# ── 3a. Average post-action throughput per condition ──────────────────────────
# The VIAVI scenario maintains aggregate throughput above the 5 Mbps operator
# intent even with aggressive sleep actions (network has sufficient capacity
# headroom). We therefore report the actual throughput maintained rather than
# a binary violation rate, which would be zero for all conditions.
tp_stats = (
    df_main.groupby('condition')['avg_throughput_mbps']
    .agg(['mean', 'std', 'count'])
    .reindex(CONDITION_ORDER)
    .reset_index()
)
tp_stats['se'] = tp_stats['std'] / np.sqrt(tp_stats['count'])

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(
    [CONDITION_LABELS[c] for c in tp_stats['condition']],
    tp_stats['mean'],
    yerr=tp_stats['se'],
    color=[PALETTE[c] for c in tp_stats['condition']],
    capsize=4, width=0.55, error_kw={'linewidth': 1.2},
)
ax.axhline(QOS_VIOLATION_THRESHOLD_MBPS, color='red', linestyle='--',
           linewidth=1.2, label=f'Operator intent ({QOS_VIOLATION_THRESHOLD_MBPS} Mbps)')
ax.set_ylabel('Avg throughput after sleep actions (Mbps)')
ax.set_title('Network Throughput Maintained by Condition')
ax.legend(fontsize=9, frameon=False)
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'analysis' / 'fig_throughput.pdf', bbox_inches='tight')
plt.show()
print(tp_stats[['condition','mean','se']].round(3).to_string(index=False))

In [ ]:
# ── 3b. Energy savings vs throughput scatter (efficiency frontier) ─────────────
fig, ax = plt.subplots(figsize=(6, 5))
for cond in CONDITION_ORDER:
    e = energy_stats.loc[energy_stats['condition'] == cond, 'mean'].values
    t = tp_stats.loc[tp_stats['condition'] == cond, 'mean'].values
    if len(e) == 0 or len(t) == 0 or np.isnan(e[0]) or np.isnan(t[0]):
        continue
    ax.scatter(e[0] * 100, t[0], color=PALETTE[cond], s=120, zorder=3)
    ax.annotate(
        CONDITION_LABELS[cond], (e[0] * 100, t[0]),
        textcoords='offset points', xytext=(6, 4), fontsize=9
    )

ax.axhline(QOS_VIOLATION_THRESHOLD_MBPS, color='red', linestyle='--',
           linewidth=1.0, label=f'Operator intent ({QOS_VIOLATION_THRESHOLD_MBPS} Mbps)')
ax.set_xlabel('Mean cells sleeping (%)')
ax.set_ylabel('Avg throughput maintained (Mbps)')
ax.set_title('Energy Savings vs Throughput (Efficiency Frontier)')
ax.xaxis.set_major_formatter(mticker.PercentFormatter())
ax.legend(fontsize=9, frameon=False)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'analysis' / 'fig_tradeoff.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ── 3c. QoS violation timeline — throughput vs intent for all conditions ───────
# Each panel shows network throughput over 96 iterations.
# Red ✗ markers = QoS violation (throughput < 5 Mbps intent).
# This makes the "cost" of each baseline visible compared to the full harness.

fig, axes = plt.subplots(len(CONDITION_ORDER), 1, figsize=(12, 9), sharex=True)

for ax, cond in zip(axes, CONDITION_ORDER):
    sub   = df_main[df_main['condition'] == cond].sort_values('iteration')
    iters = sub['iteration'].values
    tp    = sub['avg_throughput_mbps'].values
    viols = sub[sub['qos_violated'] == 1]
    n_v   = int(sub['qos_violated'].sum())

    ax.fill_between(iters, tp, alpha=0.15, color=PALETTE[cond])
    ax.plot(iters, tp, color=PALETTE[cond], linewidth=1.4)
    ax.axhline(QOS_VIOLATION_THRESHOLD_MBPS, color='red', linestyle='--',
               linewidth=1.1, alpha=0.8, label=f'Intent ({QOS_VIOLATION_THRESHOLD_MBPS} Mbps)')

    if not viols.empty:
        ax.scatter(viols['iteration'], viols['avg_throughput_mbps'],
                   color='red', s=60, zorder=5, marker='x', linewidths=2)

    ax.set_ylabel('Mbps', fontsize=9)
    ax.set_ylim(bottom=0)
    viol_label = f'{n_v} violation{"s" if n_v != 1 else ""}' if n_v > 0 else 'no violations'
    ax.set_title(f'{CONDITION_LABELS[cond]}  —  {viol_label}', fontsize=10,
                 color='red' if n_v > 0 else '#2e7d32')
    ax.legend(fontsize=8, frameon=False, loc='upper right')

axes[-1].set_xlabel('Iteration (1 iter = 15 min virtual time)')
fig.suptitle('Network Throughput vs Operator Intent (5 Mbps) — All Conditions', fontsize=12)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'analysis' / 'fig_qos_violations.pdf', bbox_inches='tight')
plt.show()

# Summary table
print('QoS violations per condition (at 5 Mbps intent):')
print(df_main.groupby('condition')['qos_violated'].sum()
      .rename(index=CONDITION_LABELS).rename('violations').to_string())

## 4. Decision quality

In [ ]:
# ── Approval rate (only meaningful for LLM conditions with a validator) ───────
_all_llm_conds = [c for c in CONDITION_ORDER if has_validator(c)]
dec_stats = (
    df_main[df_main['condition'].isin(_all_llm_conds) & df_main['n_proposed'].gt(0)]
    .groupby('condition')[['approval_rate', 'n_proposed', 'n_rejected']]
    .agg({'approval_rate': ['mean', 'std'], 'n_proposed': 'mean', 'n_rejected': 'mean'})
)
dec_stats.columns = ['approval_mean', 'approval_std', 'avg_proposed', 'avg_rejected']
llm_conds = [c for c in _all_llm_conds if c in dec_stats.index]
print('Decision quality (LLM conditions with validator):')
print(dec_stats.round(3))

if not llm_conds:
    print('⚠  No validator conditions in data yet — skipping decision quality chart.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(9, 4))

    axes[0].bar(
        [CONDITION_LABELS[c] for c in llm_conds],
        dec_stats.loc[llm_conds, 'approval_mean'] * 100,
        yerr=dec_stats.loc[llm_conds, 'approval_std'] * 100,
        color=[PALETTE[c] for c in llm_conds], capsize=4, width=0.5
    )
    axes[0].set_ylabel('Action approval rate (%)')
    axes[0].set_title('Validator Approval Rate')
    axes[0].yaxis.set_major_formatter(mticker.PercentFormatter())

    x = np.arange(len(llm_conds))
    w = 0.35
    axes[1].bar(x - w/2, dec_stats.loc[llm_conds, 'avg_proposed'],
                width=w, color='#bdbdbd', label='Proposed')
    axes[1].bar(x + w/2,
                dec_stats.loc[llm_conds, 'avg_proposed'] - dec_stats.loc[llm_conds, 'avg_rejected'],
                width=w, color=[PALETTE[c] for c in llm_conds], label='Approved')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels([CONDITION_LABELS[c] for c in llm_conds], rotation=10, ha='right')
    axes[1].set_ylabel('Actions per iteration (mean)')
    axes[1].set_title('Proposed vs Approved Actions')
    axes[1].legend(fontsize=9, frameon=False)

    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / 'analysis' / 'fig_decision_quality.pdf', bbox_inches='tight')
    plt.show()

## 4b. Fault safety and switching stability

Sleep %, throughput and violation counts do not register two things the tools change: whether a cell is put to sleep while it has an active fault, and how much on/off churn the policy causes. Each condition's approved actions are replayed against the alarm/fault feed (the same time-of-day faults apply to every condition).

In [ ]:
# ── 4b. Fault safety and switching stability ─────────────────────────────────
# Replays each condition's approved actions to rebuild which cells are asleep
# after every iteration, then checks them against the alarm/fault feed.
# The fault timeline is keyed by time of day and identical for every
# condition, so baselines without the alarm tool are scored on the same faults.
#   unsafe sleeps       — sleep actions issued on a cell with an active fault
#   faulted asleep      — cell-intervals a cell spent asleep while faulted
#                         (includes faults that start after the cell slept)
#   switches            — approved sleep + wake actions (on/off churn)
import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from tools.alarm_fault import query as fault_query

ITER_MINUTES = 15  # 1 iteration = 15 min virtual time; iteration 1 = 00:00
_DAY_START   = pd.Timestamp('2024-01-01')


def _actions(x):
    return [a for a in x if isinstance(a, dict)] if isinstance(x, list) else []


safety_rows = []
for cond in CONDITION_ORDER:
    sub = df_main[df_main['condition'] == cond].sort_values('iteration')
    if sub.empty or 'approved_actions' not in sub.columns:
        continue
    asleep: set[int] = set()
    n_sleep = n_wake = unsafe = unsafe_high = faulted_asleep = asleep_total = 0
    for _, row in sub.iterrows():
        ts     = _DAY_START + pd.Timedelta(minutes=ITER_MINUTES * (int(row['iteration']) - 1))
        faults = {cid: v for cid, v in fault_query(ts).items() if v['active']}
        for a in _actions(row['approved_actions']):
            cid = a.get('cell_id')
            if a.get('action') == 'sleep':
                n_sleep += 1
                if cid in faults:
                    unsafe      += 1
                    unsafe_high += faults[cid]['severity'] == 'high'
                asleep.add(cid)
            elif a.get('action') == 'wake':
                n_wake += 1
                asleep.discard(cid)
        faulted_asleep += len(asleep & faults.keys())
        asleep_total   += len(asleep)

    tp        = sub['avg_throughput_mbps']
    shortfall = (QOS_VIOLATION_THRESHOLD_MBPS - tp[tp < QOS_VIOLATION_THRESHOLD_MBPS])
    safety_rows.append({
        'condition':          cond,
        'sleep_actions':      n_sleep,
        'unsafe_sleeps':      unsafe,
        'unsafe_sleeps_high': unsafe_high,
        'unsafe_sleep_pct':   100 * unsafe / n_sleep if n_sleep else 0.0,
        'faulted_asleep':     faulted_asleep,
        'faulted_asleep_pct': 100 * faulted_asleep / asleep_total if asleep_total else 0.0,
        'switches':           n_sleep + n_wake,
        'mean_shortfall':     shortfall.mean() if len(shortfall) else 0.0,
        'min_tp':             tp.min(),
    })

safety_stats = pd.DataFrame(safety_rows).set_index('condition')
print('Fault safety and switching stability:')
print(safety_stats.rename(index=CONDITION_LABELS).round(2).to_string())

if safety_stats.empty:
    print('⚠  No approved_actions in results — skipping fault safety chart.')
else:
    conds  = list(safety_stats.index)
    labels = [CONDITION_LABELS[c] for c in conds]
    colors = [PALETTE[c] for c in conds]

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    panels = [
        ('unsafe_sleeps', 'Sleep actions on faulted cells', 'Unsafe Sleep Actions'),
        ('switches',      'Approved sleep + wake actions',  'Switching Churn (96 iterations)'),
    ]
    for ax, (col, ylabel, title) in zip(axes, panels):
        bars = ax.bar(labels, safety_stats[col], color=colors, width=0.55)
        ax.bar_label(bars, fmt='%d', padding=2, fontsize=9)
        ax.margins(y=0.12)  # headroom so value labels clear the title
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.tick_params(axis='x', labelrotation=15)
        plt.setp(ax.get_xticklabels(), ha='right')

    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / 'analysis' / 'fig_fault_safety.pdf', bbox_inches='tight')
    plt.show()

## 5. Ablation study

In [ ]:
# ── Ablation: each tool added to digital-twin baseline ───────────────────────
ABLATION_TOOL_MAP = {
    f'ablation_{t}': t for t in TOOL_NAMES
}

df_abl  = df[df['condition'].isin(list(ABLATION_TOOL_MAP.keys()) + ['baseline_digital_twin_vtzcycle', 'full_harness_vtzcycle'])].copy()
abl_agg = df_abl.groupby('condition')[['sleep_fraction', 'qos_violated']].mean().round(4)

abl_rows = []
if 'baseline_digital_twin_vtzcycle' in abl_agg.index:
    base_energy = abl_agg.loc['baseline_digital_twin_vtzcycle', 'sleep_fraction']
    base_qos    = abl_agg.loc['baseline_digital_twin_vtzcycle', 'qos_violated']

    for cond_key, tool_name in ABLATION_TOOL_MAP.items():
        if cond_key not in abl_agg.index:
            continue
        row = abl_agg.loc[cond_key]
        abl_rows.append({
            'tool':               tool_name,
            'sleep_fraction':     row['sleep_fraction'],
            'delta_energy_pct':   (row['sleep_fraction'] - base_energy) * 100,
            'qos_violation_rate': row['qos_violated'],
            'delta_qos_pct':      (row['qos_violated']   - base_qos)   * 100,
        })

if not abl_rows:
    print('⚠  No ablation runs found — skipping ablation chart.')
    print('   Run full_harness_vtzcycle.py with individual --tools flags to generate ablation data.')
    print('   Example: python experiments/full_harness_vtzcycle.py --tools historical_kpi --iterations 96')
else:
    df_ablation = pd.DataFrame(abl_rows).set_index('tool')

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    colors_energy = ['#1565c0' if d >= 0 else '#ef5350' for d in df_ablation['delta_energy_pct']]
    axes[0].barh(df_ablation.index, df_ablation['delta_energy_pct'], color=colors_energy)
    axes[0].axvline(0, color='black', linewidth=0.8, linestyle='--')
    axes[0].set_xlabel('Δ Energy savings vs Baseline C (pp)')
    axes[0].set_title('Marginal Energy Gain per Tool')

    colors_qos = ['#1565c0' if d <= 0 else '#ef5350' for d in df_ablation['delta_qos_pct']]
    axes[1].barh(df_ablation.index, df_ablation['delta_qos_pct'], color=colors_qos)
    axes[1].axvline(0, color='black', linewidth=0.8, linestyle='--')
    axes[1].set_xlabel('Δ QoS violation rate vs Baseline C (pp)')
    axes[1].set_title('Marginal QoS Impact per Tool (negative = better)')

    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / 'analysis' / 'fig_ablation.pdf', bbox_inches='tight')
    plt.show()
    print(df_ablation.round(4))

## 6. Tool Decision Trace (Full Harness)

For each iteration the harness saves `tool_blocks`: which cells were blocked by which tool
and which cells ended up as sleep/wake candidates. These charts make the reasoning behind
every decision visible rather than hidden inside log messages.

In [ ]:
# ── 6a. Per-iteration tool blocking timeline ──────────────────────────────────
df_fh = df[df['condition'] == 'full_harness_vtzcycle__nemotron-3-super-120b-a12b'].copy().sort_values('iteration')

has_blocks = (
    'tool_blocks' in df_fh.columns
    and df_fh['tool_blocks'].apply(lambda x: isinstance(x, dict) and bool(x)).any()
)

if not has_blocks:
    print('tool_blocks not in results — re-run full_harness_vtzcycle after the context.py update.')
else:
    def _tb(row, key):
        tb = row.get('tool_blocks', {})
        if not isinstance(tb, dict):
            return 0
        val = tb.get(key, [])
        return len(val) if isinstance(val, list) else int(bool(val))

    df_fh['n_fault_blocked']        = df_fh.apply(lambda r: _tb(r, 'fault_blocked'),        axis=1)
    df_fh['n_forecast_blocked']     = df_fh.apply(lambda r: _tb(r, 'forecast_blocked'),     axis=1)
    df_fh['n_interference_blocked'] = df_fh.apply(lambda r: _tb(r, 'interference_blocked'), axis=1)
    df_fh['n_sleep_candidates']     = df_fh.apply(lambda r: _tb(r, 'sleep_candidates'),     axis=1)
    df_fh['n_wake_candidates']      = df_fh.apply(lambda r: _tb(r, 'wake_candidates'),      axis=1)
    df_fh['surge_imminent']         = df_fh['tool_blocks'].apply(
        lambda x: int(x.get('surge_imminent', False)) if isinstance(x, dict) else 0
    )

    iters = df_fh['iteration'].values

    fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

    # — Panel 1: blocking counts per tool ————————————————————————————————————
    ax = axes[0]
    ax.fill_between(iters, df_fh['n_fault_blocked'],        alpha=0.7, label='Fault-blocked',        color='#e53935')
    ax.fill_between(iters, df_fh['n_forecast_blocked'],     alpha=0.5, label='Forecast-blocked',     color='#fb8c00')
    ax.fill_between(iters, df_fh['n_interference_blocked'], alpha=0.7, label='Interference-blocked', color='#fdd835')
    for xs in iters[df_fh['surge_imminent'].values.astype(bool)]:
        ax.axvline(xs, color='#fb8c00', linewidth=0.4, alpha=0.4)
    ax.set_ylabel('Cells blocked')
    ax.set_title('Full Harness (Nemotron 120B) — Tool Blocking per Iteration')
    ax.legend(fontsize=8, frameon=False, ncol=3)

    # — Panel 2: sleep / wake candidates ————————————————————————————————————
    ax = axes[1]
    ax.fill_between(iters, df_fh['n_sleep_candidates'], alpha=0.7, label='Sleep candidates', color='#1565c0')
    ax.fill_between(iters, df_fh['n_wake_candidates'],  alpha=0.7, label='Wake candidates',  color='#2e7d32')
    for xs in iters[df_fh['surge_imminent'].values.astype(bool)]:
        ax.axvline(xs, color='#fb8c00', linewidth=0.4, alpha=0.4)
    ax.set_ylabel('Candidate cells')
    ax.set_title('Sleep & Wake Candidates Surfaced to Planner')
    ax.legend(fontsize=8, frameon=False)

    # — Panel 3: QoS violations alongside sleep fraction ————————————————————
    ax = axes[2]
    ax2 = ax.twinx()
    ax.fill_between(iters, df_fh['sleep_fraction'] * 100, alpha=0.3, color='#1565c0', label='Sleep %')
    ax.plot(iters,          df_fh['sleep_fraction'] * 100, color='#1565c0', linewidth=1.2)
    viol_iters = iters[df_fh['qos_violated'].values.astype(bool)]
    ax2.scatter(viol_iters, [1] * len(viol_iters), marker='x', color='red', s=60, zorder=4, label='QoS violation')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Sleep fraction (%)')
    ax2.set_yticks([])
    ax.yaxis.set_major_formatter(mticker.PercentFormatter())
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8, frameon=False)

    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / 'analysis' / 'fig_tool_trace.pdf', bbox_inches='tight')
    plt.show()
    print(f'Surge-imminent iterations: {list(iters[df_fh["surge_imminent"].values.astype(bool)])}')

In [ ]:
# ── 6b. Decision trace table around surge transitions ────────────────────────
# Shows the 3 iterations before and after each surge so you can see exactly
# which tool triggered a proactive wake vs which iterations had reactive lag.
if has_blocks:
    SURGE_CENTERS = [33, 73]   # Night→Early (iter 33) and Afternoon→Evening_1 (iter 73)
    WINDOW        = 3

    rows_out = []
    for center in SURGE_CENTERS:
        lo, hi = max(1, center - WINDOW), min(df_fh['iteration'].max(), center + WINDOW)
        for _, row in df_fh[(df_fh['iteration'] >= lo) & (df_fh['iteration'] <= hi)].iterrows():
            tb    = row.get('tool_blocks', {}) or {}
            rows_out.append({
                'iter':          int(row['iteration']),
                'surge_imminent': '⚠' if tb.get('surge_imminent') else '',
                'forecast_blk':  tb.get('forecast_blocked', []),
                'fault_blk':     tb.get('fault_blocked', []),
                'interf_blk':    tb.get('interference_blocked', []),
                'sleep_cands':   tb.get('sleep_candidates', []),
                'wake_cands':    tb.get('wake_candidates', []),
                'n_approved':    int(row.get('n_approved', 0)),
                'tp_mbps':       round(row.get('avg_throughput_mbps', 0) or 0, 2),
                'qos_ok':        '✓' if not row.get('qos_violated') else '✗ VIOL',
            })

    df_trace = pd.DataFrame(rows_out)
    pd.set_option('display.max_colwidth', 40)
    print('Decision trace at surge transitions (iterations ±3 around iters 33 and 73):')
    print()
    print(df_trace.to_string(index=False))

In [ ]:
# ── 6c. Pricing-aware sleep behaviour: all conditions by pricing tier ──────────
# All runs share the same VTZ clock, so every iteration maps to the same pricing
# tier regardless of condition.  We extract the tier-per-iteration from the
# Nemotron run (which logged it) and join it onto all conditions.
# Digital twin has no pricing tool — its sleep fraction should be flat across
# tiers, making the full harness's pricing-aware modulation clearly visible.

TIER_ORDER  = ["off_peak", "shoulder", "peak"]
TIER_COLORS = {"off_peak": "#90caf9", "shoulder": "#ffe082", "peak": "#ef9a9a"}
TIER_LABELS = {"off_peak": "Off-peak\n($0.08/kWh)", "shoulder": "Shoulder\n($0.14/kWh)", "peak": "Peak\n($0.21–0.25/kWh)"}

# Build iter → pricing_tier map from the Nemotron run
def _get_tier(row):
    if "pricing_tier" in row and row["pricing_tier"]:
        return row["pricing_tier"]
    tb = row.get("tool_blocks", {})
    if isinstance(tb, dict):
        return tb.get("pricing_tier", "")
    return ""

nemotron_tiers = (
    df[df["condition"] == "full_harness_vtzcycle__nemotron-3-super-120b-a12b"]
    .assign(pricing_tier=lambda d: d.apply(_get_tier, axis=1))
    [["iteration", "pricing_tier"]]
    .query("pricing_tier != ''")
    .drop_duplicates("iteration")
    .set_index("iteration")["pricing_tier"]
)

if nemotron_tiers.empty:
    print("pricing_tier not found in Nemotron results — cannot derive tier map.")
else:
    # Apply the tier map to all main conditions
    df_pricing = df_main.copy()
    df_pricing["pricing_tier"] = df_pricing["iteration"].map(nemotron_tiers)
    df_pricing = df_pricing[df_pricing["pricing_tier"].isin(TIER_ORDER)]

    plot_conds = [c for c in CONDITION_ORDER if c in df_pricing["condition"].unique()]
    n = len(plot_conds)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4), sharey=True)
    if n == 1:
        axes = [axes]

    for ax, cond in zip(axes, plot_conds):
        sub   = df_pricing[df_pricing["condition"] == cond]
        means = sub.groupby("pricing_tier")["sleep_fraction"].mean().reindex(TIER_ORDER) * 100
        errs  = sub.groupby("pricing_tier")["sleep_fraction"].std().reindex(TIER_ORDER) * 100

        ax.bar([TIER_LABELS[t] for t in TIER_ORDER], means, yerr=errs,
               color=[TIER_COLORS[t] for t in TIER_ORDER],
               capsize=4, width=0.5, error_kw={"linewidth": 1.2})
        ax.set_title(CONDITION_LABELS.get(cond, cond), fontsize=9)
        ax.set_ylabel("Mean cells sleeping (%)" if ax is axes[0] else "")
        ax.yaxis.set_major_formatter(mticker.PercentFormatter())
        ax.set_xlabel("Energy pricing tier")

    fig.suptitle(
        "Sleep Fraction by Pricing Tier\n"
        "Full Harness modulates with price; Digital Twin is flat (no pricing tool)",
        fontsize=11
    )
    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / "analysis" / "fig_pricing_sleep.pdf", bbox_inches="tight")
    plt.show()

    pivot = (
        df_pricing.groupby(["condition", "pricing_tier"])["sleep_fraction"]
        .mean().mul(100).round(1).unstack("pricing_tier").reindex(columns=TIER_ORDER)
    )
    pivot.index = [CONDITION_LABELS.get(c, c) for c in pivot.index]
    print("Mean sleep % by pricing tier:")
    print(pivot.to_string())

In [ ]:
# ── 6d. Planner pricing adherence — did the LLM act on the pricing signal? ────
# Inspects the 'reason' text in proposed_actions for full_harness iterations
# where pricing_tier == 'peak'.  If the planner is following STEP 2.5 it will
# write reasons containing "peak pricing".  Reports adherence rate so you can
# benchmark Qwen 7B vs Nemotron without manually reading the JSONL.

df_fh_p = df[df["condition"] == "full_harness_vtzcycle__nemotron-3-super-120b-a12b"].copy()

has_pricing2 = (
    "pricing_tier" in df_fh_p.columns
    and df_fh_p["pricing_tier"].ne("").any()
)

if not has_pricing2:
    # fallback: try extracting from tool_blocks
    def _get_tier2(row):
        tb = row.get("tool_blocks", {})
        if isinstance(tb, dict):
            return tb.get("pricing_tier", "")
        return ""
    df_fh_p["pricing_tier"] = df_fh_p.apply(_get_tier2, axis=1)
    has_pricing2 = df_fh_p["pricing_tier"].ne("").any()

if not has_pricing2:
    print("pricing_tier not in results — re-run full_harness after latest code pull.")
else:
    peak_iters = df_fh_p[df_fh_p["pricing_tier"] == "peak"]
    total_peak  = len(peak_iters)

    # Count sleep actions where reason mentions pricing
    pricing_keywords = ["peak pricing", "peak price", "cost saving", "pricing"]

    def _pricing_reason(actions):
        if not isinstance(actions, list):
            return 0, 0
        sleeps = [a for a in actions if isinstance(a, dict) and a.get("action") == "sleep"]
        priced = [a for a in sleeps
                  if any(kw in str(a.get("reason", "")).lower() for kw in pricing_keywords)]
        return len(priced), len(sleeps)

    peak_iters = peak_iters.copy()
    peak_iters[["n_pricing_reasons", "n_sleep_actions"]] = peak_iters["proposed_actions"].apply(
        lambda x: pd.Series(_pricing_reason(x))
    )

    total_sleeps  = peak_iters["n_sleep_actions"].sum()
    priced_sleeps = peak_iters["n_pricing_reasons"].sum()
    iters_with_any_pricing = (peak_iters["n_pricing_reasons"] > 0).sum()

    adherence_pct = 100 * priced_sleeps / total_sleeps if total_sleeps > 0 else 0
    iter_pct      = 100 * iters_with_any_pricing / total_peak if total_peak > 0 else 0

    print("=== Planner Pricing Adherence (full_harness, peak-pricing iterations) ===")
    print(f"  Peak-pricing iterations : {total_peak}")
    print(f"  Sleep actions proposed  : {int(total_sleeps)}")
    print(f"  Actions citing pricing  : {int(priced_sleeps)}  ({adherence_pct:.0f}% of sleep actions)")
    print(f"  Iterations with ≥1 pricing reason: {iters_with_any_pricing} / {total_peak}  ({iter_pct:.0f}%)")
    print()
    if adherence_pct >= 70:
        print("✓  Planner is reliably following the peak-pricing instruction.")
    elif adherence_pct >= 30:
        print("⚠  Planner is inconsistently following the peak-pricing instruction.")
        print("   Consider switching to a larger model (e.g. Nemotron 120B).")
    else:
        print("✗  Planner is NOT following the peak-pricing instruction.")
        print("   Qwen 7B is likely ignoring STEP 2.5 — switch to Nemotron 120B/A12B.")

In [ ]:
# ── 6c. Tool signal correlation with agent decisions ─────────────────────────
# Uses tool_blocks (now saved to JSONL) rather than tool_context (was never serialised).
df_fh2 = df[df['condition'] == 'full_harness_vtzcycle__nemotron-3-super-120b-a12b'].copy()

has_blocks2 = (
    'tool_blocks' in df_fh2.columns
    and df_fh2['tool_blocks'].apply(lambda x: isinstance(x, dict) and bool(x)).any()
)

if not has_blocks2:
    print('tool_blocks not in results — re-run full_harness_vtzcycle after the context.py update.')
else:
    def _corr_signals(row):
        tb = row.get('tool_blocks', {}) or {}
        return pd.Series({
            'n_fault_blocked':        len(tb.get('fault_blocked', [])        or []),
            'n_forecast_blocked':     len(tb.get('forecast_blocked', [])     or []),
            'n_interference_blocked': len(tb.get('interference_blocked', []) or []),
            'n_sleep_candidates':     len(tb.get('sleep_candidates', [])     or []),
            'surge_imminent':         int(tb.get('surge_imminent', False)),
        })

    signals2 = df_fh2.apply(_corr_signals, axis=1)
    df_corr2 = pd.concat(
        [df_fh2[['sleep_fraction', 'n_approved', 'qos_violated']].reset_index(drop=True),
         signals2.reset_index(drop=True)],
        axis=1
    ).dropna()

    corr_matrix = df_corr2.corr(method='spearman')
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(
        corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
        center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.5
    )
    ax.set_title('Spearman Correlation: Tool Signals vs Agent Decisions (Nemotron 120B)')
    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / 'analysis' / 'fig_tool_correlation.pdf', bbox_inches='tight')
    plt.show()

## 7. Statistical significance

In [ ]:
# ── Mann-Whitney U: full harness vs each baseline ────────────────────────────
fh_energy = df_main[df_main['condition'] == 'full_harness_vtzcycle__nemotron-3-super-120b-a12b']['sleep_fraction'].values
fh_qos    = df_main[df_main['condition'] == 'full_harness_vtzcycle__nemotron-3-super-120b-a12b']['qos_violated'].values

sig_rows = []
for cond in ['baseline_rules', 'baseline_openloop_vtzcycle', 'baseline_digital_twin_vtzcycle']:
    sub = df_main[df_main['condition'] == cond]
    if sub.empty:
        sig_rows.append({
            'vs': CONDITION_LABELS[cond],
            'energy_U': '—', 'energy_p': '—', 'energy_sig': '(no data)',
            'qos_U':    '—', 'qos_p':    '—', 'qos_sig':    '(no data)',
        })
        continue
    u_e, p_e = stats.mannwhitneyu(fh_energy, sub['sleep_fraction'].values, alternative='greater')
    u_q, p_q = stats.mannwhitneyu(fh_qos,    sub['qos_violated'].values,   alternative='less')
    sig_rows.append({
        'vs':          CONDITION_LABELS[cond],
        'energy_U':    int(u_e),
        'energy_p':    round(p_e, 4),
        'energy_sig':  '✓' if p_e < 0.05 else '✗',
        'qos_U':       int(u_q),
        'qos_p':       round(p_q, 4),
        'qos_sig':     '✓' if p_q < 0.05 else '✗',
    })

df_sig = pd.DataFrame(sig_rows).set_index('vs')
print('Mann-Whitney U — Full Harness (Nemotron 120B) vs baselines (one-sided):')
print('  energy: full_harness > baseline (H_a)')
print('  qos:    full_harness < baseline violations (H_a)')
print()
print(df_sig.to_string())

## 8. Summary table (paper-ready)

In [ ]:
summary_rows = []
for cond in CONDITION_ORDER:
    sub = df_main[df_main['condition'] == cond]
    validated = has_validator(cond)
    planner   = sub['planner_model'].iloc[0]   if 'planner_model'   in sub.columns else ''
    validator = sub['validator_model'].iloc[0] if 'validator_model' in sub.columns else ''
    summary_rows.append({
        'Condition':            CONDITION_LABELS[cond],
        'Planner':              planner.split('/')[-1]   if isinstance(planner, str) and planner   else ('rule' if cond == 'baseline_rules' else 'not recorded'),
        'Validator':            validator.split('/')[-1] if isinstance(validator, str) and validator else ('not recorded' if validated else 'N/A'),
        'Mean sleep %':         f"{sub['sleep_fraction'].mean() * 100:.1f} ± {sub['sleep_fraction'].std() * 100:.1f}",
        'QoS viol. %':          f"{sub['qos_violated'].mean() * 100:.1f}",
        'Avg tp (Mbps)':        f"{sub['avg_throughput_mbps'].mean():.2f}",
        'Approval rate':        f"{sub['approval_rate'].mean() * 100:.1f}%" if validated else 'N/A',
        'Avg latency (s)':      f"{sub['total_elapsed_s'].mean():.1f}" if 'total_elapsed_s' in sub.columns else 'N/A',
        'Unsafe sleeps':        f"{safety_stats.loc[cond, 'unsafe_sleeps']} ({safety_stats.loc[cond, 'unsafe_sleep_pct']:.0f}%)" if cond in safety_stats.index else 'N/A',
        'Switches':             f"{safety_stats.loc[cond, 'switches']}" if cond in safety_stats.index else 'N/A',
        'Min tp (Mbps)':        f"{sub['avg_throughput_mbps'].min():.2f}",
        'Mean shortfall (Mbps)': f"{safety_stats.loc[cond, 'mean_shortfall']:.2f}" if cond in safety_stats.index else 'N/A',
        'Faulted asleep %':     f"{safety_stats.loc[cond, 'faulted_asleep_pct']:.1f}" if cond in safety_stats.index else 'N/A',
    })

df_summary = pd.DataFrame(summary_rows).set_index('Condition')
print('=== PAPER SUMMARY TABLE ===')
print(df_summary.to_string())

# Save as CSV for LaTeX import
df_summary.to_csv(PROJECT_ROOT / 'analysis' / 'table_summary.csv')
print('\n✓ Saved table_summary.csv  (import into LaTeX with \\input or pandas to_latex)')

In [ ]:
# ── LaTeX table snippet ───────────────────────────────────────────────────────
print('LaTeX table snippet:\n')
print(df_summary.to_latex(caption='Performance comparison across experimental conditions.',
                           label='tab:results', escape=False))

## 9. Policy tuning: mock prediction vs VIAVI

The harness thresholds (PRB sleep gates, throughput headroom, surge ratio, overload limit) are tuned offline on the mock RSG by `experiments/tune_policy.py`, then each tuned policy is run once on VIAVI. Two versions of the mock were used:

- **Analytic mock**: a hand-anchored radio formula. `policies/tuned.yaml` was tuned on it.
- **VIAVI-calibrated mock**: throughput and per-cell PRB fitted to logged VIAVI runs (`mock_rsg/calibrate.py`). `policies/tuned_calibrated.yaml` was tuned on it.

For each policy and intent the table puts the reward predicted by each mock next to the reward measured on VIAVI. Each intent with a VIAVI run of a tuned policy gets its own rows. The harness decisions are deterministic: in every run the Validator approved all candidates. So a mock that matches VIAVI predicts the run exactly, and the md5 of the approved actions shows whether it did.

Reward = mean fraction of N1 cells asleep − 1.0 × avoidable QoS violation rate − 0.001 × switches per iteration (`experiments/reward.py`). A violation is avoidable when cells were asleep while throughput was below intent.

In [ ]:
# ── 9a. Score the policy runs and compare with the mock predictions ─────────
import contextlib, hashlib, io, os, sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from agent.policy import load_policy
from experiments.full_harness_vtzcycle import run as run_harness
from experiments.reward import episode_metrics
from experiments.tune_policy import FallbackPlanner, RuleValidator
from mock_rsg.scenario import Scenario as MockScenario

ITER_MINUTES = 15                       # 1 iteration = 15 min of virtual time
_DAY_START   = pd.Timestamp('2024-01-01')   # iteration 1 = 00:00
_MODEL = 'nemotron-3-super-120b-a12b'
POLICY_RUNS = {   # policy file → condition name of its VIAVI run
    'default':          f'full_harness_vtzcycle__{_MODEL}',
    'tuned':            f'full_harness_vtzcycle_tuned__{_MODEL}',
    'tuned_calibrated': f'full_harness_vtzcycle_tuned_calibrated__{_MODEL}',
}
POLICY_LABELS = {
    'default':          'Default thresholds',
    'tuned':            'Tuned on analytic mock',
    'tuned_calibrated': 'Tuned on calibrated mock',
}


def _decision_md5(iterations):
    dec = json.dumps([sorted((a['action'], a['cell_id']) for a in it.get('approved_actions', []))
                      for it in iterations])
    return hashlib.md5(dec.encode()).hexdigest()


def _viavi_iterations(condition, intent):
    """Raw iterations of the latest run of a condition at an intent (as loaded in section 1)."""
    dirs = df_all_raw.loc[(df_all_raw['condition'] == condition) & (df_all_raw['intent_mbps'] == intent),
                          'run_dir'].unique()
    if not len(dirs):
        return None
    its = load_jsonl(OUTPUT_DIR / sorted(dirs)[-1] / 'iterations.jsonl')
    for it in its:   # older logs have no timestamp; iteration 1 = 00:00
        it.setdefault('timestamp', str(_DAY_START + pd.Timedelta(minutes=ITER_MINUTES * (it['iteration'] - 1))))
    return its


def _mock_iterations(policy, calibration, intent):
    """One deterministic 96-step episode on the mock, rule stand-ins for the LLMs."""
    old = os.environ.get('MOCK_RSG_CALIBRATION')
    os.environ['MOCK_RSG_CALIBRATION'] = calibration   # '' = calibrated file, 'none' = analytic
    try:
        scenario = MockScenario(str(PROJECT_ROOT / 'ai_rsg_config' / 'config.conf'), 'mock://')
        with contextlib.redirect_stdout(io.StringIO()):
            return run_harness(scenario, f'{intent:g} Mbps', 96, policy=policy,
                               llm=FallbackPlanner(), validator_llm=RuleValidator(), verbose=False)
    finally:
        if old is None:
            os.environ.pop('MOCK_RSG_CALIBRATION', None)
        else:
            os.environ['MOCK_RSG_CALIBRATION'] = old


# Intents with a VIAVI run of a tuned policy (always including the main intent)
_policy_runs = df_all_raw[df_all_raw['condition'].isin(list(POLICY_RUNS.values())[1:])]
INTENTS = sorted({MAIN_INTENT_MBPS, *_policy_runs['intent_mbps'].unique()}, reverse=True)

tuning_rows = []
for intent in INTENTS:
    for name, cond in POLICY_RUNS.items():
        path  = PROJECT_ROOT / 'policies' / f'{name}.yaml'
        viavi = _viavi_iterations(cond, intent)
        if not path.exists() or viavi is None:
            print(f'⚠  skipping {name} at {intent:g} Mbps: '
                  + ('no policy file' if not path.exists() else f'no VIAVI run of {cond}'))
            continue
        policy     = load_policy(path)
        m          = episode_metrics(viavi, intent)
        analytic   = _mock_iterations(policy, 'none', intent)
        calibrated = _mock_iterations(policy, '', intent)
        viavi_md5  = _decision_md5(viavi)
        tuning_rows.append({
            'intent_mbps':        intent,
            'policy':             name,
            'pred_analytic':      episode_metrics(analytic, intent)['reward'],
            'pred_calibrated':    episode_metrics(calibrated, intent)['reward'],
            'viavi_reward':       m['reward'],
            'sleep_pct':          100 * m['mean_sleep_fraction'],
            'cost_saving_pct':    100 * episode_metrics(viavi, intent, objective='cost')['cost_weighted_saving'],
            'qos_viol_pct':       100 * m['qos_violation_rate'],
            'avoidable_viol_pct': 100 * m['avoidable_violation_rate'],
            'switches':           m['switches'],
            'rejections':         sum(len(it.get('rejected_actions') or []) for it in viavi),
            'calibrated_exact':   _decision_md5(calibrated) == viavi_md5,
            'decisions_md5':      viavi_md5,
        })

    # Digital twin baseline for context (no tunable policy, so no mock prediction)
    _dt = _viavi_iterations('baseline_digital_twin_vtzcycle', intent)
    if _dt is not None:
        m = episode_metrics(_dt, intent)
        tuning_rows.append({
            'intent_mbps': intent, 'policy': 'digital_twin', 'viavi_reward': m['reward'],
            'sleep_pct': 100 * m['mean_sleep_fraction'],
            'cost_saving_pct': 100 * episode_metrics(_dt, intent, objective='cost')['cost_weighted_saving'],
            'qos_viol_pct': 100 * m['qos_violation_rate'], 'avoidable_viol_pct': 100 * m['avoidable_violation_rate'],
            'switches': m['switches'], 'rejections': sum(len(it.get('rejected_actions') or []) for it in _dt),
        })
POLICY_LABELS['digital_twin'] = 'C: Digital twin (no policy)'

tuning_stats = pd.DataFrame(tuning_rows).set_index(['intent_mbps', 'policy'])
_default = tuning_stats.xs('default', level='policy')['viavi_reward'] if tuning_rows else pd.Series(dtype=float)
tuning_stats['vs_default'] = [r - _default.get(i, np.nan) for (i, _), r in tuning_stats['viavi_reward'].items()]

# Two narrower tables, so neither wraps
_lab = tuning_stats.rename(index=POLICY_LABELS, level='policy')
print('Reward on VIAVI, and what each mock predicted:')
print(_lab[['pred_analytic', 'pred_calibrated', 'viavi_reward', 'vs_default', 'calibrated_exact']].round(4).to_string())
print('\nOperating metrics on VIAVI:')
print(_lab[['sleep_pct', 'cost_saving_pct', 'qos_viol_pct', 'avoidable_viol_pct', 'switches', 'rejections']]
      .round(2).to_string())


In [ ]:
# ── 9b. Predicted vs measured reward per policy, one panel per intent ────────
_intents = [i for i in INTENTS if (i, 'default') in tuning_stats.index or
            any((i, p) in tuning_stats.index for p in POLICY_RUNS)]
if not _intents:
    print('⚠  No policy runs found — skipping chart.')
else:
    series = [
        ('pred_analytic',   'Predicted: analytic mock',   '#bdbdbd'),
        ('pred_calibrated', 'Predicted: calibrated mock', '#90caf9'),
        ('viavi_reward',    'Measured on VIAVI',          '#1565c0'),
    ]
    fig, axes = plt.subplots(1, len(_intents), figsize=(8 * len(_intents), 4.2), squeeze=False, sharey=True)
    for ax, intent in zip(axes[0], _intents):
        sub  = tuning_stats.xs(intent, level='intent_mbps')
        _pol = [p for p in POLICY_RUNS if p in sub.index]
        x, w = np.arange(len(_pol)), 0.26
        for k, (col, label, color) in enumerate(series):
            vals = sub.loc[_pol, col].astype(float)
            bars = ax.bar(x + (k - 1) * w, vals, w - 0.02, label=label, color=color)
            ax.bar_label(bars, fmt='%.3f', padding=2, fontsize=8)
        if 'digital_twin' in sub.index:
            dt = sub.loc['digital_twin', 'viavi_reward']
            ax.axhline(dt, color='#616161', lw=1, ls='--')
            ax.text(len(_pol) - 0.5, dt, f'Digital twin on VIAVI ({dt:.3f})', ha='right', va='bottom',
                    fontsize=8, color='#424242')
        ax.set_xticks(x, [POLICY_LABELS[p] for p in _pol])
        ax.set_title(f'{intent:g} Mbps intent')
        ax.grid(axis='y', alpha=0.3)
    axes[0][0].set_ylabel('Episode reward')
    axes[0][0].set_ylim(0, max(0.85, axes[0][0].get_ylim()[1]))
    axes[0][0].legend(frameon=False, fontsize=9, loc='upper left')
    fig.suptitle('Policy tuning: mock prediction vs VIAVI')
    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / 'analysis' / 'fig_policy_tuning.pdf', bbox_inches='tight')
    plt.show()


### Findings

- **The analytic mock ranked the policies in the wrong order, so tuning on it made the policy worse on VIAVI.** It underestimates how many cells can sleep (it predicts 0.362 for the default, against 0.645 measured) and scored `tuned` above the default (0.415 vs 0.362). On VIAVI the order is reversed: `tuned`'s 1.27 throughput headroom blocks every sleep at 09:00–13:00 and 20:00–22:00, giving 0.517 against 0.645. It also rated `tuned_calibrated`, the best policy on VIAVI, the worst of the three (0.351).
- **The calibrated mock reproduces all three VIAVI runs exactly**: the same approved actions in every iteration (same md5), and so the same reward. It uses the throughput VIAVI logged for each set of sleeping cells, and leaves out runs from older VIAVI setups. Before the `tuned_calibrated` run was added, it predicted 0.714 against 0.704 measured. VIAVI had never been run with 14 cells asleep at that load, and its throughput there (5.11 and 5.10 Mbps at iterations 39 and 83) was just under the 5.15 Mbps guard, so the 15th sleep waited until the next window. Each VIAVI run adds the states it visits, so the mock reproduces it from then on.
- **Tuning on the calibrated mock beat the default on VIAVI**: +0.059 reward (+9%), 6 percentage points more cells asleep (71.5% against 65.5%) and a cost-weighted saving of 45.6% against 41.2%. QoS violations are the same as the default (12.5%, all with every cell awake), with no avoidable violations and no Validator rejections.
- **The digital-twin baseline sleeps about as many cells as `tuned_calibrated` (70.9% against 71.5%), but not safely**: 6.25% avoidable QoS violations (cells asleep while throughput was below intent), 6 Validator rejections and 180 switches against 103. Its reward is the lowest (0.378).

`calibrated_exact` is computed against the current `mock_rsg/viavi_calibration.json`; recalibrate (`python -m mock_rsg.calibrate output/run_*`) after new VIAVI runs.